In [16]:
import os
import pandas as pd
import cv2

randomseed = 42
printDebug = True

frames1 = pd.read_csv('kaggle_dataset/labels_train.csv')
frames2 = pd.read_csv('kaggle_dataset/labels_trainval.csv')
frames3 = pd.read_csv('kaggle_dataset/labels_val.csv')
frames = pd.concat([frames1, frames2, frames3], ignore_index=True)

# araba olmayan frame'leri sil
frames = frames[frames['class_id'] == 1]

frames.drop_duplicates(inplace=True)


def calculate_bounding_box_ratio(row, threshold = 0.2):
    width = row['xmax'] - row['xmin']
    height = row['ymax'] - row['ymin']
    image = cv2.imread("kaggle_dataset/images/" + row['frame'])
    h, w, _ = image.shape
    w_ratio = width / w
    h_ratio = height / h
    ratio = max(w_ratio, h_ratio)
    return ratio >= threshold

frames = frames[frames.apply(calculate_bounding_box_ratio, axis=1)]

if printDebug:
    print(frames.shape)



(13155, 6)


In [17]:
# bounding bocları yeniden hesapla

def recalculate_bounding_boxes(imgH, imgW, row):
    xmin = int(row['xmin'] / imgW *256)
    xmax = int(row['xmax'] / imgW *256)
    ymin = int(row['ymin'] / imgH *256)
    ymax = int(row['ymax'] / imgH *256)
    return xmin, xmax, ymin, ymax



for i, row in frames.iterrows():
    img = cv2.imread("kaggle_dataset/images/" + row['frame'])
    imgH, imgW, _ = img.shape
    xmin, xmax, ymin, ymax = recalculate_bounding_boxes(imgH, imgW, row)
    frames.loc[i, ['xmin', 'xmax', 'ymin', 'ymax']] = [xmin, xmax, ymin, ymax]

if printDebug:
    #print max and min values of bounding boxes
    print("xmin max: ", frames['xmin'].max())
    print("xmax max: ", frames['xmax'].max())
    print("ymin max: ", frames['ymin'].max())
    print("ymax max: ", frames['ymax'].max())


xmin max:  251
xmax max:  255
ymin max:  234
ymax max:  255


In [21]:
# aynı bölgede birden fazla araba varsa frame yi sil

regions = []

for j in range(0, 3):
    for i in range(0, 3):
        regions.append([i * 64, (i + 1) * 64 + 64, j * 64, (j + 1) * 64 + 64])

def calculate_iou(region, bounding_box, threshold = 0.1):
    rx1, rx2, ry1, ry2 = region
    bx1, bx2, by1, by2 = bounding_box

    xmin = max(rx1, bx1)
    xmax = min(rx2, bx2)
    ymin = max(ry1, by1)
    ymax = min(ry2, by2)

    # kesişim yok
    if xmax <= xmin or ymax <= ymin:
        return False

    region_area = (rx2 - rx1) * (ry2 - ry1)
    bounding_box_area = (bx2 - bx1) * (by2 - by1)
    intersection_area = max(0, xmax - xmin) * max(0, ymax - ymin)

    iou = intersection_area / (region_area + bounding_box_area - intersection_area)

    return iou >= threshold

# Her bölgede araba olup olmadığı ile ilgili yeni bir sütun ekle
frames['y'] = [[0] * 9 for _ in range(len(frames))]


for i, row in frames.iterrows():
    bounding_box = [row['xmin'], row['xmax'], row['ymin'], row['ymax']]
    y = frames.loc[i, 'y']
    for j, region in enumerate(regions):
        if calculate_iou(region, bounding_box):
            y[j] = 1

    frames.at[i, 'y'] = y

# aynı frame için y değerini topla, y girdilerinden herhangi biri 1'den büyükse frame sil
# y listesini 9 ayrı sütuna ayır
y_df = pd.DataFrame(frames['y'].tolist(), index=frames.index)

# frame bazında topla
y_sum = y_df.groupby(frames['frame']).sum()

# Herhangi bir sütunda değer > 1 olan frameler
frames_to_remove = y_sum[y_sum.gt(1).any(axis=1)].index

# Bu frameleri sil
frames = frames[~frames['frame'].isin(frames_to_remove)].reset_index(drop=True)        





if printDebug:
    for _, row in frames.iterrows():
        y = row['y']
        image = cv2.imread("kaggle_dataset/images/" + row['frame'])
        image = cv2.resize(image,(256, 256))
        cv2.rectangle(image,(row['xmin'], row['ymin']), (row['xmax'], row['ymax']), (255, 0, 0), 2)
        for j, region in enumerate(regions):
            if y[j] == 1:
                cv2.rectangle(image, (region[0], region[2]), (region[1], region[3]), (0, 255, 0), 2)
            else:
                cv2.rectangle(image, (region[0], region[2]), (region[1], region[3]), (0, 0, 255), 2)
        cv2.imshow('image', image)
        cv2.waitKey(0)
        key = cv2.waitKey(0) & 0xFF
        if key == 27:  # ESC key to break
            break
    
    cv2.destroyAllWindows()
    print(f"frames shape: {frames.shape}")
    print("regions: ", regions)
    print("frames: ", frames['y'].max())


frames shape: (7502, 7)
regions:  [[0, 128, 0, 128], [64, 192, 0, 128], [128, 256, 0, 128], [0, 128, 64, 192], [64, 192, 64, 192], [128, 256, 64, 192], [0, 128, 128, 256], [64, 192, 128, 256], [128, 256, 128, 256]]
frames:  [1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
# csv kaydet

# %70 dateset train, %15 val, %15 test // Not: Aynı frameler hem train hem val hem de test setinde çıkabilir. Duplicate frameleri atınca 6500 resim kalıyor ondan yapmadım
frames_train = frames.sample(frac=0.7, random_state=randomseed)
frames_val = frames[~frames.index.isin(frames_train.index)].sample(frac=0.5, random_state=randomseed)
frames_test = frames[~frames.index.isin(pd.concat([frames_train, frames_val]).index)]

frames_train.to_csv("frames_train.csv", index=False)
frames_val.to_csv("frames_val.csv", index=False)
frames_test.to_csv("frames_test.csv", index=False)